In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
"""Auto-generated by arc_prize_2026.arc3.kaggle.build_notebook — do not edit; edit the sources in src/."""
from __future__ import annotations
import random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from typing import Any
import numpy as np
from arcengine import FrameData, GameAction, GameState


# ---- perception.py ----
"""Frame -> objects -> abstract state key.

An ARC-AGI-3 frame is a 64x64 grid of colours 0..15. We segment it into same-colour 4-connected
components. The background is the largest component (by area). Objects carry colour, bbox, area and a
shape hash (mask relative to its bbox), so a moved object keeps its identity and a recoloured object
keeps its shape. The abstract state key is the sorted tuple of (colour, y0, x0, shape_hash) for all
non-background objects; two frames with the same key are the same abstract state.
"""



H = W = 64


@dataclass(frozen=True)
class Obj:
    colour: int
    y0: int
    x0: int
    y1: int  # exclusive
    x1: int
    area: int
    shape: int  # hash of the boolean mask inside the bbox
    mask: bytes = field(default=b"", compare=False, hash=False, repr=False)  # packed bool mask, for rendering

    @property
    def h(self) -> int:
        return self.y1 - self.y0

    @property
    def w(self) -> int:
        return self.x1 - self.x0

    @property
    def centre(self) -> tuple[int, int]:
        return ((self.y0 + self.y1 - 1) // 2, (self.x0 + self.x1 - 1) // 2)


def label_components(grid: np.ndarray) -> tuple[np.ndarray, int]:
    """4-connected same-colour labelling. Returns (labels, n) with labels in 1..n."""
    h, w = grid.shape
    labels = np.zeros((h, w), dtype=np.int32)
    n = 0
    for y in range(h):
        row = grid[y]
        for x in range(w):
            if labels[y, x]:
                continue
            n += 1
            c = row[x]
            stack = [(y, x)]
            labels[y, x] = n
            while stack:
                cy, cx = stack.pop()
                if cy > 0 and not labels[cy - 1, cx] and grid[cy - 1, cx] == c:
                    labels[cy - 1, cx] = n; stack.append((cy - 1, cx))
                if cy < h - 1 and not labels[cy + 1, cx] and grid[cy + 1, cx] == c:
                    labels[cy + 1, cx] = n; stack.append((cy + 1, cx))
                if cx > 0 and not labels[cy, cx - 1] and grid[cy, cx - 1] == c:
                    labels[cy, cx - 1] = n; stack.append((cy, cx - 1))
                if cx < w - 1 and not labels[cy, cx + 1] and grid[cy, cx + 1] == c:
                    labels[cy, cx + 1] = n; stack.append((cy, cx + 1))
    return labels, n


def segment(grid: np.ndarray, drop_background: bool = True) -> list[Obj]:
    grid = np.asarray(grid, dtype=np.int16)
    labels, n = label_components(grid)
    objs = []
    flat = labels.ravel()
    order = np.argsort(flat, kind="stable")
    sorted_labels = flat[order]
    bounds = np.searchsorted(sorted_labels, np.arange(1, n + 2))
    ys_all, xs_all = np.divmod(order, grid.shape[1])
    for k in range(n):
        idx = slice(bounds[k], bounds[k + 1])
        ys, xs = ys_all[idx], xs_all[idx]
        y0, y1, x0, x1 = int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1
        mask = np.zeros((y1 - y0, x1 - x0), dtype=bool)
        mask[ys - y0, xs - x0] = True
        objs.append(Obj(int(grid[ys[0], xs[0]]), y0, x0, y1, x1, int(len(ys)), hash(mask.tobytes()) ^ (mask.shape[0] << 20) ^ mask.shape[1], mask.tobytes()))
    if drop_background and objs:
        bg = max(objs, key=lambda o: o.area)
        objs = [o for o in objs if o is not bg]
    objs.sort(key=lambda o: (o.y0, o.x0, o.colour, o.shape))
    return objs


def ident(o: Obj) -> tuple[int, int, int]:
    """Object class used for learning effects: colour and bbox size (robust to small occlusions)."""
    return (o.colour, o.h, o.w)


def state_key(objs: list[Obj]) -> int:
    """Identity by colour + bbox; a hole carved into a floor by a sprite does not change the floor."""
    return hash(tuple((o.colour, o.y0, o.x0, o.y1, o.x1) for o in objs))


def frame_key(grid) -> int:
    return hash(np.asarray(grid, dtype=np.int8).tobytes())


def diff(before: list[Obj], after: list[Obj], max_disp: int = 32) -> dict:
    """Object-level diff. same = equal colour+bbox; moved = same class (colour,h,w), nearest within max_disp;
    recoloured = equal bbox, different colour; the rest appeared/vanished."""
    b_left, a_left = list(before), list(after)
    same, moved_, recol = [], [], []
    a_index = {(p.colour, p.y0, p.x0, p.y1, p.x1): p for p in a_left}
    for o in list(b_left):
        p = a_index.get((o.colour, o.y0, o.x0, o.y1, o.x1))
        if p is not None and p in a_left:
            same.append(o); b_left.remove(o); a_left.remove(p)
    for o in list(b_left):
        best = None
        for p in a_left:
            if ident(o) == ident(p):
                d = abs(p.y0 - o.y0) + abs(p.x0 - o.x0)
                if d <= max_disp and (best is None or d < best[0]):
                    best = (d, p)
        if best is not None:
            p = best[1]; moved_.append((o, p, (p.y0 - o.y0, p.x0 - o.x0))); b_left.remove(o); a_left.remove(p)
    for o in list(b_left):
        for p in list(a_left):
            if (o.y0, o.x0, o.y1, o.x1) == (p.y0, p.x0, p.y1, p.x1):
                recol.append((o, p)); b_left.remove(o); a_left.remove(p); break
    return {"same": same, "moved": moved_, "recoloured": recol, "vanished": b_left, "appeared": a_left}

def moved(o: Obj, dy: int, dx: int) -> Obj:
    return Obj(o.colour, o.y0 + dy, o.x0 + dx, o.y1 + dy, o.x1 + dx, o.area, o.shape, o.mask)


def render_move(grid: np.ndarray, o: Obj, new: Obj, bg: int) -> np.ndarray:
    """Return a copy of grid with object o erased (filled with bg) and redrawn at new's position."""
    g = grid.copy()
    m = np.frombuffer(o.mask, dtype=bool).reshape(o.h, o.w) if o.mask else np.ones((o.h, o.w), dtype=bool)
    sub = g[o.y0:o.y1, o.x0:o.x1]; sub[m] = bg
    H, W = g.shape
    if 0 <= new.y0 and new.y1 <= H and 0 <= new.x0 and new.x1 <= W:
        sub2 = g[new.y0:new.y1, new.x0:new.x1]; sub2[m] = o.colour
    return g


def background_colour(grid: np.ndarray) -> int:
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[counts.argmax()])

# ---- effects.py ----
"""Online object-level action-effect model.

Observations are (objects_before, action_key, objects_after). We learn, per action key:
  * `moves[key]`      : Counter of (colour, shape) -> Counter of displacement (dy, dx) — which object class
                        moves and by how much when this action is taken (per-object deltas from `diff`).
  * `blocked[key]`    : Counter of colours seen immediately ahead (in the learned move direction) of the
                        moving object on steps where it did NOT move (no-op for that object).
  * `passable[key]`   : Counter of colours ahead on steps where it DID move.
  * `transitions`     : exact memo (state_key, action_key) -> next state_key (deterministic games).
  * `effects[key]`    : Counter of coarse effect signatures (moved/recoloured/appeared/vanished counts),
                        used to decide whether an action does anything at all in general.
`predict(objs, key)` returns the predicted object list (or None if unknown), by applying the modal
displacement to the modal moving class when the cells ahead are background-or-passable; it returns the
same objects (no-op) when the cells ahead are in the blocked set. Only movement effects are predicted
generatively; recolour/appear/vanish are memorised through `transitions` and otherwise count as unknown.
"""




Key = object  # action key: int action id, or ("click", y//4, x//4)


def _ahead_cells(o: Obj, dy: int, dx: int, grid: np.ndarray, members: list[Obj] | None = None) -> list[int]:
    """Colours of every cell the object sweeps through when displaced by (dy, dx) in unit steps,
    excluding cells currently occupied by any member of its rigid group. [-1] marks the world edge."""
    H, W = grid.shape
    occupied = members or [o]
    steps = max(abs(dy), abs(dx))
    if steps == 0:
        return []
    sy, sx = (dy > 0) - (dy < 0), (dx > 0) - (dx < 0)
    cells: list[int] = []
    for k in range(1, steps + 1):
        ty0, tx0 = o.y0 + sy * k, o.x0 + sx * k
        ty1, tx1 = ty0 + o.h, tx0 + o.w
        if ty0 < 0 or tx0 < 0 or ty1 > H or tx1 > W:
            return [-1]
        for y in range(ty0, ty1):
            for x in range(tx0, tx1):
                if any(m.y0 <= y < m.y1 and m.x0 <= x < m.x1 for m in occupied):
                    continue
                cells.append(int(grid[y, x]))
    return cells


DEATH = -1  # sentinel next-state key for transitions that ended the game


class EffectModel:
    def __init__(self) -> None:
        self.moves: dict[Key, Counter] = defaultdict(Counter)          # key -> Counter[((colour, shape), (dy, dx))]
        self.blocked: dict[Key, Counter] = defaultdict(Counter)
        self.passable: dict[Key, Counter] = defaultdict(Counter)
        self.transitions: dict[tuple[int, Key], int] = {}
        self.effects: dict[Key, Counter] = defaultdict(Counter)
        self.deaths: Counter = Counter()
        # click effects by clicked class: ckey -> Counter[signature]; signature in
        #   ("none",) | ("recolour", new_colour) | ("vanish",) | ("other",)
        self.click_effects: dict[Key, Counter] = defaultdict(Counter)
        # side effects on other classes when clicking ckey: ckey -> Counter[(other_ident, "recolour", new_colour)]
        self.click_side: dict[Key, Counter] = defaultdict(Counter)
        self.n_obs: Counter = Counter()
        self.n_pred = 0
        self.n_pred_correct = 0
        self.key_pred: dict[Key, deque] = defaultdict(lambda: deque(maxlen=20))  # last 20 generative outcomes per key
        self.min_reliability = 0.5

    # ---- learning -------------------------------------------------------------------------------
    def observe(self, before: list[Obj], grid_before: np.ndarray, key: Key, after: list[Obj], clicked: Obj | None = None) -> dict:
        d = diff(before, after)
        if clicked is not None and isinstance(key, tuple) and key[0] == "click":
            self._observe_click(key, clicked, d, before, after)
        sk_b, sk_a = state_key(before), state_key(after)
        self.transitions[(sk_b, key)] = sk_a
        self.n_obs[key] += 1
        sig = (len(d["moved"]), len(d["recoloured"]), len(d["appeared"]), len(d["vanished"]))
        self.effects[key][sig] += 1
        for o, p, delta in d["moved"]:
            self.moves[key][(ident(o), delta)] += 1
            self.passable[key].update(_ahead_cells(o, *delta, grid_before))
        group = self.modal_group(key)
        if group is not None and not d["moved"]:
            # the group did not move (HUD/bar changes elsewhere are irrelevant): record what was in its way
            idents, (dy, dx) = group
            members = [o for o in before if ident(o) in idents]
            if members and all(any((m.colour, m.y0, m.x0, m.y1, m.x1) == (p.colour, p.y0, p.x0, p.y1, p.x1) for p in after) for m in members):
                for o in members:
                    self.blocked[key].update(_ahead_cells(o, dy, dx, grid_before, members))
        return d

    def _observe_click(self, ckey: Key, clicked: Obj, d: dict, before: list[Obj], after: list[Obj]) -> None:
        same_pos = lambda a, b: (a.y0, a.x0, a.y1, a.x1) == (b.y0, b.x0, b.y1, b.x1)
        sig = ("none",)
        if any(same_pos(o, clicked) for o in d["same"]) and not (d["moved"] or d["recoloured"] or d["vanished"] or d["appeared"]):
            sig = ("none",)
        else:
            rec = [(o, p) for o, p in d["recoloured"] if same_pos(o, clicked)]
            if rec:
                sig = ("recolour", rec[0][1].colour)
            elif any(same_pos(o, clicked) for o in d["vanished"]):
                sig = ("vanish",)
            elif any(same_pos(o, clicked) for o in d["same"]):
                sig = ("other",)  # clicked object unchanged, something else changed
            else:
                sig = ("other",)
        self.click_effects[ckey][sig] += 1
        for o, p in d["recoloured"]:
            if not same_pos(o, clicked):
                self.click_side[ckey][(ident(o), "recolour", p.colour)] += 1

    def modal_click(self, ckey: Key):
        c = self.click_effects.get(ckey)
        if not c:
            return None
        sig, n = c.most_common(1)[0]
        return sig if n / sum(c.values()) >= 0.6 and sum(c.values()) >= 2 else None

    def predict_click(self, objs: list[Obj], ckey: Key, clicked: Obj) -> tuple[list[Obj] | None, str]:
        """Generative prediction for a click on `clicked` (class ckey) from its modal signature."""
        sig = self.modal_click(ckey)
        if sig is None:
            return None, "unknown"
        hist = self.key_pred[ckey]
        src = "unreliable" if (len(hist) >= 10 and sum(hist) / len(hist) < self.min_reliability) else "model"
        if sig[0] == "none":
            return list(objs), src
        out = []
        for o in objs:
            if o is clicked:
                if sig[0] == "recolour":
                    out.append(Obj(sig[1], o.y0, o.x0, o.y1, o.x1, o.area, o.shape, o.mask))
                elif sig[0] == "vanish":
                    continue
                else:
                    out.append(o)
            else:
                out.append(o)
        if sig[0] == "other":
            # consistent side effects (>= 80% of clicks on this class) are applied; else unknown
            total = sum(self.click_effects[ckey].values())
            side = {k[0]: k[2] for k, n in self.click_side[ckey].items() if n >= 0.8 * total and k[1] == "recolour"}
            if not side:
                return None, "unknown"
            out = [Obj(side[ident(o)], o.y0, o.x0, o.y1, o.x1, o.area, o.shape, o.mask) if ident(o) in side else o for o in out]
        return out, src

    def modal_group(self, key: Key):
        """(set of idents that move together, modal displacement) or None."""
        if not self.moves[key]:
            return None
        by_delta = Counter()
        for (cls, delta), n in self.moves[key].items():
            by_delta[delta] += n
        delta, _ = by_delta.most_common(1)[0]
        counts = {cls: n for (cls, d), n in self.moves[key].items() if d == delta}
        top = max(counts.values())
        idents = frozenset(cls for cls, n in counts.items() if n >= 0.5 * top)
        return idents, delta

    def is_known_noop(self, key: Key) -> bool:
        c = self.effects[key]
        return sum(c.values()) >= 3 and c[(0, 0, 0, 0)] / sum(c.values()) > 0.9

    # ---- prediction -----------------------------------------------------------------------------
    def predict(self, objs: list[Obj], grid: np.ndarray, key: Key) -> tuple[list[Obj] | None, str]:
        """Return (predicted objects or None, source in {memo, model, unknown})."""
        sk = state_key(objs)
        if (sk, key) in self.transitions:
            return None, "memo"
        group = self.modal_group(key)
        if group is None:
            return None, "unknown"
        hist = self.key_pred[key]
        unreliable = len(hist) >= 10 and sum(hist) / len(hist) < self.min_reliability
        idents, (dy, dx) = group
        members = [o for o in objs if ident(o) in idents]
        if not members:
            return None, "unknown"
        bl, pa = self.blocked[key], self.passable[key]
        blocked = False
        for o in members:
            for c in _ahead_cells(o, dy, dx, grid, members):
                if c == -1 or (bl[c] > 0 and bl[c] >= pa[c]):
                    blocked = True; break
            if blocked:
                break
        src = "unreliable" if unreliable else "model"
        if blocked:
            return list(objs), src
        out = [_moved(o, dy, dx) if ident(o) in idents else o for o in objs]
        return out, src

    def predict_unchecked(self, objs: list[Obj], key: Key) -> list[Obj] | None:
        """Apply the modal group displacement with no blocking check and no reliability gate (for goal examples)."""
        group = self.modal_group(key)
        if group is None:
            return None
        idents, (dy, dx) = group
        if not any(ident(o) in idents for o in objs):
            return None
        return [_moved(o, dy, dx) if ident(o) in idents else o for o in objs]

    def next_key(self, sk: int, key: Key) -> int | None:
        return self.transitions.get((sk, key))

    def record_death(self, sk: int, key: Key) -> None:
        self.transitions[(sk, key)] = DEATH
        self.deaths[key] += 1

    def is_deadly(self, sk: int, key: Key) -> bool:
        return self.transitions.get((sk, key)) == DEATH

    def record_prediction(self, predicted_key: int | None, observed_key: int, key: Key | None = None) -> None:
        if predicted_key is None:
            return
        self.n_pred += 1
        ok = int(predicted_key == observed_key)
        self.n_pred_correct += ok
        if key is not None:
            self.key_pred[key].append(ok)

    @property
    def accuracy(self) -> float | None:
        return self.n_pred_correct / self.n_pred if self.n_pred else None

# ---- goals.py ----
"""Goal inference from level transitions.

When `levels_completed` increments, the state just before the transition is a positive example of the
level's goal; the level's initial state is a negative example. We enumerate predicates from a fixed
template class over object classes (ident = (colour, h, w)) and keep those that are TRUE at the
transition and FALSE at the level start. Across levels of one game, surviving predicates are those
that held at every transition seen (intersection). Later levels then plan toward any surviving predicate.

Templates:
  Gone(c)         : no object of class c remains                      (collect/clear everything of c)
  Count(c, k)     : exactly k objects of class c remain, k < initial  (partial clearing)
  Touch(c)        : the mover group's bbox touches an object of class c (reach c)
  Overlap(c, d)   : an object of class c overlaps an object of class d (bring c onto d)
  Appeared(c)     : class c absent at start, present at the end       (reveal/create c)
"""




def _touch(a: Obj, b: Obj) -> bool:
    return a.y0 <= b.y1 and b.y0 <= a.y1 and a.x0 <= b.x1 and b.x0 <= a.x1


def _overlap(a: Obj, b: Obj) -> bool:
    return a.y0 < b.y1 and b.y0 < a.y1 and a.x0 < b.x1 and b.x0 < a.x1


@dataclass(frozen=True)
class Predicate:
    kind: str
    a: tuple
    b: tuple | int | None = None

    def holds(self, objs: list[Obj], mover_idents: frozenset) -> bool:
        if self.kind == "gone":
            return not any(ident(o) == self.a for o in objs)
        if self.kind == "count":
            return sum(ident(o) == self.a for o in objs) == self.b
        if self.kind == "appeared":
            return any(ident(o) == self.a for o in objs)
        if self.kind == "touch":
            members = [o for o in objs if ident(o) in mover_idents]
            targets = [o for o in objs if ident(o) == self.a]
            return any(_touch(m, t) for m in members for t in targets)
        if self.kind == "overlap":
            xs = [o for o in objs if ident(o) == self.a]; ys = [o for o in objs if ident(o) == self.b]
            return any(_overlap(x, y) for x in xs for y in ys)
        return False

    def classes(self) -> set:
        return {self.a} | ({self.b} if isinstance(self.b, tuple) else set())

    def __str__(self) -> str:
        return f"{self.kind}({self.a}{', ' + str(self.b) if self.b is not None else ''})"


MAX_PREDICATE_AREA = 300  # objects this large are scenery, not goal participants


def is_bar(o: Obj) -> bool:
    """Thin, long objects are HUD bars / borders, never goal participants."""
    return (o.h <= 2 and o.w >= 20) or (o.w <= 2 and o.h >= 20)


def goal_participants(objs: list[Obj]) -> list[Obj]:
    return [o for o in objs if o.h * o.w < MAX_PREDICATE_AREA and not is_bar(o)]


def candidate_predicates(init_objs: list[Obj], final_objs: list[Obj], mover_idents: frozenset) -> set[Predicate]:
    init_objs = goal_participants(init_objs)
    final_objs = goal_participants(final_objs)
    ci, cf = Counter(ident(o) for o in init_objs), Counter(ident(o) for o in final_objs)
    classes = set(ci) | set(cf)
    cands: set[Predicate] = set()
    for c in classes:
        if c in mover_idents:
            continue
        if ci[c] > 0 and cf[c] == 0:
            cands.add(Predicate("gone", c))
        if ci[c] > cf[c] > 0:
            cands.add(Predicate("count", c, cf[c]))
        if ci[c] == 0 and cf[c] > 0:
            cands.add(Predicate("appeared", c))
    if mover_idents:
        for c in classes - set(mover_idents):
            p = Predicate("touch", c)
            if p.holds(final_objs, mover_idents) and not p.holds(init_objs, mover_idents):
                cands.add(p)
    non_mover = [c for c in classes if c not in mover_idents]
    for c in non_mover:
        for d in non_mover:
            if c >= d:
                continue
            p = Predicate("overlap", c, d)
            if p.holds(final_objs, mover_idents) and not p.holds(init_objs, mover_idents):
                cands.add(p)
    return cands


class GoalMemory:
    def __init__(self) -> None:
        self.surviving: set[Predicate] | None = None
        self.transitions = 0

    def update(self, cands: set[Predicate]) -> None:
        self.transitions += 1
        self.surviving = set(cands) if self.surviving is None else (self.surviving & cands)

    def active(self, objs: list[Obj]) -> list[Predicate]:
        """Surviving predicates whose classes exist (or, for gone/count, existed) in the current level."""
        if not self.surviving:
            return []
        present = {ident(o) for o in objs}
        out = []
        for p in self.surviving:
            if p.kind in ("gone", "count", "touch") and p.a not in present:
                continue
            if p.kind == "overlap" and not (p.a in present and p.b in present):
                continue
            out.append(p)
        order = {"touch": 0, "overlap": 1, "count": 2, "gone": 3, "appeared": 4}
        return sorted(out, key=lambda p: order[p.kind])

# ---- planner.py ----
"""Breadth-first planning in the learned effect model.

Nodes carry (state_key, objs, grid). Edges: for every candidate action key, the model predicts the next
objects (memo edges are followed by key only, which is enough to recognise known states but not to
expand them further unless we also saw their objects; we keep an objs cache per state key for that).
Target: the nearest state holding an untried (state, key) pair whose key class is not a known no-op.
Returns the action-key path to that state plus the exploratory key to take there.
"""





def adjacent_classes(members: list[Obj], objs: list[Obj]) -> set:
    """Classes of non-member objects whose bbox touches (8-neighbourhood) any member's bbox."""
    out = set()
    for m in members:
        for o in objs:
            if o in members:
                continue
            if o.y0 <= m.y1 and m.y0 <= o.y1 and o.x0 <= m.x1 and m.x0 <= o.x1:
                out.add(ident(o))
    return out


def plan_to_information(model: EffectModel, objs: list[Obj], grid: np.ndarray, keys_for, tried: dict[int, set],
                        objs_cache: dict[int, tuple[list[Obj], np.ndarray]], max_depth: int = 12, max_nodes: int = 1500,
                        mover_idents: frozenset | None = None, contacted: dict | None = None, goal_test=None,
                        click_model: bool = False):
    """keys_for(objs) -> list of (key, class_key, action, data). Returns (path_keys, final_key) or (None, None).

    Target priority: (1) a reachable state where the mover group touches an object class contacted fewer
    times than any other (contact-driven curiosity), taking an untried key there; (2) the nearest state with
    an untried, non-no-op key, preferring keys with the fewest observations."""
    bg = background_colour(grid)
    start = state_key(objs)
    objs_cache.setdefault(start, (objs, grid))
    q = deque([(start, [])])
    seen = {start}
    nodes = 0
    best = None
    pred_cache: dict[tuple, tuple] = {}
    contact_best = None  # (contact_count, depth, path, key)
    min_contact = min(contacted.values()) if contacted else 0
    while q and nodes < max_nodes:
        sk, path = q.popleft()
        nodes += 1
        if sk not in objs_cache:
            continue
        o_list, g = objs_cache[sk]
        if goal_test is not None and path and goal_test(o_list):
            return path, None  # goal state reached in the model: execute the path, no exploratory key needed
        cands = keys_for(o_list, len(path))
        untried = [c for c in cands if c[0] not in tried.get(sk, set()) and not model.is_known_noop(c[1]) and not model.is_deadly(sk, c[0])]
        if untried:
            untried.sort(key=lambda c: (model.n_obs[c[1]], ))
        # contact-driven target: mover touches a rarely-contacted class
        if mover_idents and contacted is not None and untried:
            members = [o for o in o_list if ident(o) in mover_idents]
            if members:
                adj = adjacent_classes(members, o_list)
                if adj:
                    cc = min(contacted.get(c, 0) for c in adj)
                    if contact_best is None or cc < contact_best[0]:
                        contact_best = (cc, len(path), path, untried[0])
                        if cc <= min_contact and path:
                            return path, untried[0]
        if untried and path and contact_best is None and best is None:
            best = (path, untried[0])
        if untried and not path and best is None:
            best = ([], untried[0])
        if len(path) >= max_depth:
            continue
        for key, ckey, action, data in cands:
            nk = model.next_key(sk, key)
            if nk == DEATH:
                continue
            if nk is not None:
                if nk not in seen:
                    seen.add(nk); q.append((nk, path + [(key, action, data)]))
                continue
            ck = (sk, key)
            if ck in pred_cache:
                pred, src = pred_cache[ck]
            elif click_model and isinstance(key, tuple) and key[0] == "click" and key[1] != "lattice":
                clicked = next((o for o in o_list if (o.colour, o.h, o.w, o.y0, o.x0) == (key[1], key[2], key[3], key[4], key[5])), None)
                pred, src = model.predict_click(o_list, ckey, clicked) if clicked is not None else (None, "unknown")
                pred_cache[ck] = (pred, src)
            else:
                pred, src = model.predict(o_list, g, ckey)
                pred_cache[ck] = (pred, src)
            if pred is None or src == "unreliable":
                continue
            nk = state_key(pred)
            if nk == sk or nk in seen:
                continue
            # render the predicted grid: moved objects for movement predictions; recolour/erase for clicks
            g2 = g
            if len(pred) == len(o_list):
                for a, b in zip(o_list, pred):
                    if (a.y0, a.x0) != (b.y0, b.x0):
                        g2 = render_move(g2, a, b, bg)
                    elif a.colour != b.colour:
                        g2 = g2.copy(); sub = g2[a.y0:a.y1, a.x0:a.x1]; m = np.frombuffer(a.mask, dtype=bool).reshape(a.h, a.w) if a.mask else np.ones((a.h, a.w), bool); sub[m] = b.colour
            else:
                g2 = g2.copy()
                gone = [a for a in o_list if a not in pred]
                for a in gone:
                    sub = g2[a.y0:a.y1, a.x0:a.x1]; m = np.frombuffer(a.mask, dtype=bool).reshape(a.h, a.w) if a.mask else np.ones((a.h, a.w), bool); sub[m] = bg
            objs_cache.setdefault(nk, (pred, g2))
            seen.add(nk); q.append((nk, path + [(key, action, data)]))
    if contact_best is not None and contact_best[2]:
        return contact_best[2], contact_best[3]
    return best if best else (None, None)

# ---- agents/base.py ----



ACTION_BY_ID = {a.value: a for a in GameAction}


class BaseAgent:
    """Minimal stand-in for agents.agent.Agent with the same two abstract methods."""

    MAX_ACTIONS: int = 2000

    def __init__(self, game_id: str, seed: int = 0, **kwargs: Any) -> None:
        self.game_id = game_id
        self.rng = random.Random(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        raise NotImplementedError

    # helpers shared by agents -------------------------------------------------------------
    @staticmethod
    def legal_actions(latest_frame: FrameData) -> list[GameAction]:
        ids = latest_frame.available_actions or [1, 2, 3, 4, 5, 6, 7]
        return [ACTION_BY_ID[i] for i in ids if i in ACTION_BY_ID and i != 0]

    def random_action(self, latest_frame: FrameData) -> GameAction:
        a = self.rng.choice(self.legal_actions(latest_frame))
        if a.is_complex():
            a.set_data({"x": self.rng.randint(0, 63), "y": self.rng.randint(0, 63)})
        return a

# ---- agents/model_based.py ----
"""H1 agent: learn an object-level effect model online, plan in it toward information gain, keep the
model across levels (memory=True) or reset it at each level (memory=False)."""




class ModelBasedAgent(BaseAgent):
    def __init__(self, game_id: str, seed: int = 0, memory: bool = True, max_depth: int = 12, max_nodes: int = 1500,
                 click_lattice: int = 8, contact: bool = True, goals: bool = True, click_model: bool = False, **kwargs):
        super().__init__(game_id, seed)
        self.memory = memory
        self.contact = contact
        self.goals = goals
        self.click_model = click_model  # E12: generative click effects in planning (failed its gate; off by default)
        self.sequences = kwargs.get("sequences", False)  # E13: replay the level-1 clicked-class sequence on later levels
        self.seq_k = kwargs.get("seq_k", 4)
        self.level_clicks: list = []      # clicked classes in the current level (consecutive repeats compressed)
        self.learned_seq: list | None = None
        self.seq_queue: list = []          # classes still to click on this level
        self.n_seq_steps = 0
        self.goal_memory = GoalMemory()
        self.level_init_objs: list[Obj] | None = None
        self.level_init_grid: np.ndarray | None = None
        self.n_goal_plans = 0
        self.max_depth, self.max_nodes, self.click_lattice = max_depth, max_nodes, click_lattice
        self.model = EffectModel()
        self.tried: dict[int, set] = {}
        self.objs_cache: dict[int, tuple[list[Obj], np.ndarray]] = {}
        self.plan: list = []
        self.predicted_next: int | None = None
        self.prev: tuple[list[Obj], np.ndarray, object, object] | None = None  # objs, grid, key, ckey
        self.levels_seen = 0
        self.n_plans = 0; self.n_plan_steps = 0; self.n_replans_on_error = 0; self.n_fallback = 0; self.n_deaths = 0
        self.pred_by_source = {"memo": [0, 0], "model": [0, 0], "unreliable": [0, 0]}
        self.row_changes = np.zeros(64, dtype=np.int32); self.row_steps = 0; self.prev_grid: np.ndarray | None = None
        self.row_px = np.zeros(64, dtype=np.int64); self.row_maxpx = np.zeros(64, dtype=np.int64); self.mover_rows = np.zeros(64, dtype=bool)
        self.volatile_rows: np.ndarray = np.zeros(64, dtype=bool)
        self.terrain_colours: set[int] = set()
        self.contacted: dict = {}  # object class -> number of steps the mover group has touched it

    def _update_volatility(self, grid: np.ndarray, after_reset: bool) -> None:
        """Two HUD detectors feed `volatile_rows`:
        (a) rows changing on >80% of effective steps (step counters, animations);
        (b) thin progress bars: rows whose changes are tiny (<= 1 px per changing step, averaged), have
            occurred on >= 4 steps, and are never rows where the learned mover group has been seen.
        Both are measured, not assumed; `stats()` reports which rows were masked."""
        if self.prev_grid is not None and not after_reset:
            d = grid != self.prev_grid
            changed_rows = d.any(axis=1)
            if changed_rows.any():
                self.row_changes += changed_rows; self.row_steps += 1
                px = d.sum(axis=1)
                self.row_px += px
                self.row_maxpx = np.maximum(self.row_maxpx, px)
            if self.row_steps >= 10:
                rate_rows = (self.row_changes / self.row_steps) > 0.8
            else:
                rate_rows = np.zeros(64, dtype=bool)
            # thin bars: every change in the row is exactly 1 px, seen >= 6 times, never a mover row, and the
            # flagged rows form bands of at most 2 adjacent rows (a 1-px sprite wandering over many rows is not a bar)
            cand = (self.row_changes >= 6) & (self.row_maxpx <= 1) & ~self.mover_rows
            bar_rows = np.zeros(64, dtype=bool)
            y = 0
            while y < 64:
                if cand[y]:
                    y2 = y
                    while y2 + 1 < 64 and cand[y2 + 1]:
                        y2 += 1
                    if y2 - y + 1 <= 2 and (y2 <= 5 or y >= 50):  # HUD bars live at the frame edges
                        bar_rows[y:y2 + 1] = True
                    y = y2 + 1
                else:
                    y += 1
            self.volatile_rows = rate_rows | bar_rows
        self.prev_grid = grid

    TERRAIN_FRAC = 0.20  # a colour covering >= 20% of the frame is terrain (background, floor): kept in the grid for
                         # blocking checks but its objects are excluded from the state key

    def perceive(self, grid: np.ndarray) -> list[Obj]:
        objs = segment(grid, drop_background=False)
        counts = np.bincount(grid.ravel(), minlength=16)
        terrain = {int(c) for c in np.where(counts >= self.TERRAIN_FRAC * grid.size)[0]}
        if not terrain:
            terrain = {int(counts.argmax())}
        self.terrain_colours = terrain
        if self.volatile_rows.any():
            objs = [o for o in objs if not self.volatile_rows[o.y0:o.y1].all()]
        return [o for o in objs if o.colour not in terrain]

    # ---- candidate actions ---------------------------------------------------------------------
    MAX_CLICK_OBJECTS = 16  # per node: clicks on at most this many objects (largest first), lattice only if no objects

    def _keys_for(self, latest: FrameData):
        legal = self.legal_actions(latest)

        def keys(objs: list[Obj], depth: int = 0):
            out = []
            for a in legal:
                if a.is_complex():
                    # v7 behaviour (click_model off): every object at every depth; E12: cap look-ahead nodes for speed
                    cands = list(objs) if (depth == 0 or not self.click_model) else sorted(objs, key=lambda o: -o.area)[: self.MAX_CLICK_OBJECTS]
                    for o in cands:
                        cy, cx = o.centre
                        out.append((("click",) + ident(o) + (o.y0, o.x0), ("click",) + ident(o), a, {"x": int(cx), "y": int(cy)}))
                    if not cands or depth == 0 or not self.click_model:
                        step = 64 // self.click_lattice
                        for y in range(step // 2, 64, step):
                            for x in range(step // 2, 64, step):
                                out.append((("click", "lattice", y, x), ("click", "lattice"), a, {"x": x, "y": y}))
                else:
                    out.append((a.value, a.value, a, {}))
            return out
        return keys

    # ---- main -------------------------------------------------------------------------------------
    def choose_action(self, frames: list[FrameData], latest: FrameData) -> GameAction:
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            if latest.state is GameState.GAME_OVER and self.prev is not None:
                p_objs, p_grid, p_key, p_ckey = self.prev
                self.model.record_death(state_key(p_objs), p_key)   # never take this edge again
                self.n_deaths += 1
            self.prev = None; self.plan = []; self.predicted_next = None
            return GameAction.RESET
        grid = np.asarray(latest.frame[-1], dtype=np.int16)
        self._update_volatility(grid, after_reset=self.prev is None)
        objs = self.perceive(grid)
        early_movers = frozenset().union(*[g[0] for k in self.model.moves if (g := self.model.modal_group(k))]) if self.model.moves else frozenset()
        if early_movers:
            members = [o for o in objs if ident(o) in early_movers]
            # decorations (direction markers, eyes) fully inside a mover member's bbox are part of the mover: drop them
            objs = [o for o in objs if o in members or not any(m.y0 <= o.y0 and o.y1 <= m.y1 and m.x0 <= o.x0 and o.x1 <= m.x1 for m in members)]
        sk = state_key(objs)
        if self.level_init_objs is None:
            self.level_init_objs = objs; self.level_init_grid = grid
        self.objs_cache[sk] = (objs, grid)

        # E13: level transition -> remember the clicked-class sequence of the level just finished
        if self.sequences and latest.levels_completed > self.levels_seen:
            if self.level_clicks:
                self.learned_seq = self.level_clicks[-self.seq_k:]
            self.level_clicks = []
            self.seq_queue = list(self.learned_seq or [])
        # level transition bookkeeping
        if latest.levels_completed > self.levels_seen:
            if self.goals and self.prev is not None and self.level_init_grid is not None:
                mover = frozenset().union(*[g[0] for k in self.model.moves if (g := self.model.modal_group(k))]) if self.model.moves else frozenset()
                # positive example = the state the winning action produced in the model (the pre-transition
                # frame does not yet show the mover on the target); negative = level start, both re-perceived
                # under the current HUD/terrain mask so stale bar objects cannot become "goals"
                p_objs, p_grid, p_key, p_ckey = self.prev
                pre = self.perceive(p_grid)
                post, _src = self.model.predict(pre, p_grid, p_ckey)
                if post is None or state_key(post) == state_key(pre):
                    post = self.model.predict_unchecked(pre, p_ckey)
                positive = post if post is not None else pre
                init = self.perceive(self.level_init_grid)
                self.goal_memory.update(candidate_predicates(init, positive, mover))
            self.level_init_objs = objs; self.level_init_grid = grid
            self.levels_seen = latest.levels_completed
            self.tried = {}; self.plan = []; self.predicted_next = None; self.contacted = {}
            if not self.memory:
                self.model = EffectModel(); self.objs_cache = {sk: (objs, grid)}
            self.prev = None

        # learn from the last transition
        if self.prev is not None:
            p_objs, p_grid, p_key, p_ckey = self.prev
            clicked = None
            if isinstance(p_key, tuple) and p_key[0] == "click" and p_key[1] != "lattice":
                clicked = next((o for o in p_objs if (o.colour, o.h, o.w, o.y0, o.x0) == (p_key[1], p_key[2], p_key[3], p_key[4], p_key[5])), None)
            self.model.observe(p_objs, p_grid, p_ckey, objs, clicked=clicked)
            self.model.transitions[(state_key(p_objs), p_key)] = sk
            if self.predicted_next is not None:
                src = self._pred_src
                self.pred_by_source[src][0] += 1
                self.pred_by_source[src][1] += int(self.predicted_next == sk)
                self.model.record_prediction(self.predicted_next, sk, p_ckey if src in ("model", "unreliable") else None)
                if self.predicted_next != sk and src != "unreliable":
                    self.plan = []; self.n_replans_on_error += 1
            self.predicted_next = None

        mover_idents = frozenset().union(*[g[0] for k in self.model.moves if (g := self.model.modal_group(k))]) if self.model.moves else frozenset()
        if mover_idents:
            members = [o for o in objs if ident(o) in mover_idents]
            for m in members:
                self.mover_rows[m.y0:m.y1] = True
            for c in adjacent_classes(members, objs):
                self.contacted[c] = self.contacted.get(c, 0) + 1
            for o in objs:
                self.contacted.setdefault(ident(o), 0)
        keys_for = self._keys_for(latest)
        if self.sequences and self.seq_queue and not self.plan:
            legal = self.legal_actions(latest)
            click_action = next((a for a in legal if a.is_complex()), None)
            target = None
            while self.seq_queue and target is None:   # skip classes absent on this level
                cls = self.seq_queue.pop(0)
                target = next((o for o in objs if ident(o) == cls), None)
            if click_action is not None and target is not None:
                cy, cx = target.centre
                key = ("click",) + ident(target) + (target.y0, target.x0); ckey = ("click",) + ident(target)
                self.tried.setdefault(sk, set()).add(key)
                self.prev = (objs, grid, key, ckey); self.predicted_next = None; self.n_seq_steps += 1
                self.level_clicks.append(cls) if not self.level_clicks or self.level_clicks[-1] != cls else None
                click_action.set_data({"x": int(cx), "y": int(cy)})
                return click_action
        if not self.plan:
            goal_test = None
            if self.goals:
                active = self.goal_memory.active(objs)
                if active:
                    mi = mover_idents
                    goal_test = lambda o, _a=active, _m=mi: any(p.holds(o, _m) for p in _a)
            path, final = plan_to_information(self.model, objs, grid, keys_for, self.tried, self.objs_cache,
                                              self.max_depth, self.max_nodes, mover_idents or None,
                                              self.contacted if (mover_idents and self.contact) else None, goal_test,
                                              click_model=self.click_model)
            if path is not None:
                if final is None:
                    self.n_goal_plans += 1
                self.plan = list(path) + ([final[:1] + (final[2], final[3])] if final else [])
                # normalise entries to (key, action, data)
                self.plan = [(e[0], e[-2], e[-1]) for e in self.plan]
                self.n_plans += 1
        if self.plan:
            key, action, data = self.plan.pop(0)
            self.n_plan_steps += 1
        else:
            self.n_fallback += 1
            cands = keys_for(objs)
            safe = [c for c in cands if not self.model.is_deadly(sk, c[0])] or cands
            untried = [c for c in safe if c[0] not in self.tried.get(sk, set())]
            pool = untried or safe
            key, ckey, action, data = self.rng.choice(pool)
        ckey = key if not isinstance(key, tuple) else (("click",) + tuple(key[1:4]) if key[1] != "lattice" else ("click", "lattice"))
        self.tried.setdefault(sk, set()).add(key)
        # predict for accuracy accounting
        nk = self.model.next_key(sk, key)
        if nk is not None:
            self.predicted_next, self._pred_src = nk, "memo"
        else:
            if self.click_model and isinstance(key, tuple) and key[0] == "click" and key[1] != "lattice":
                clicked = next((o for o in objs if (o.colour, o.h, o.w, o.y0, o.x0) == (key[1], key[2], key[3], key[4], key[5])), None)
                pred, src = self.model.predict_click(objs, ckey, clicked) if clicked is not None else (None, "unknown")
            else:
                pred, src = self.model.predict(objs, grid, ckey)  # v7 path: clicks may move objects (displacement model)
            if pred is not None:
                self.predicted_next, self._pred_src = state_key(pred), src
        self.prev = (objs, grid, key, ckey)
        if action.is_complex():
            action.set_data(data)
            if self.sequences and isinstance(key, tuple) and key[1] != "lattice":
                cls = ("click",) + tuple(key[1:4])
                cls = tuple(key[1:4])
                if not self.level_clicks or self.level_clicks[-1] != cls:
                    self.level_clicks.append(cls)
        return action

    def stats(self) -> dict:
        return {"memory": self.memory, "contact": self.contact, "goals": self.goals, "click_model": self.click_model, "sequences": self.sequences, "seq_steps": self.n_seq_steps, "learned_seq": [str(c) for c in (self.learned_seq or [])], "plans": self.n_plans, "goal_plans": self.n_goal_plans, "deaths": self.n_deaths,
                "surviving_goals": sorted(map(str, self.goal_memory.surviving)) if self.goal_memory.surviving else None, "plan_steps": self.n_plan_steps,
                "replans_on_error": self.n_replans_on_error, "fallback_steps": self.n_fallback,
                "pred_total": self.model.n_pred, "pred_acc": None if self.model.accuracy is None else round(self.model.accuracy, 3),
                "pred_by_source": {k: (v[0], round(v[1] / v[0], 3) if v[0] else None) for k, v in self.pred_by_source.items()},
                "movers": {str(k): [sorted(map(str, g[0])), list(g[1])] for k in self.model.moves if (g := self.model.modal_group(k))},
                "volatile_rows": [int(i) for i in np.where(self.volatile_rows)[0]], "terrain_colours": sorted(self.terrain_colours),
                "known_noops": [str(k) for k in self.model.effects if self.model.is_known_noop(k)],
                "click_classes_modelled": sum(1 for k in self.model.click_effects if self.model.modal_click(k) is not None),
                "key_reliability": {str(k): [len(v), round(sum(v) / len(v), 2)] for k, v in self.model.key_pred.items() if len(v)}}

# ======================================================================================
# Kaggle framework adapter (ARC-AGI-3-Agents contract)
# ======================================================================================
import os as _os, time as _time
from agents.agent import Agent as _FrameworkAgent

_START = _time.time()
_BUDGET_S = float(_os.getenv("ARC3_BUDGET_S", "30600"))
_MAX_ACTIONS = int(_os.getenv("ARC3_MAX_ACTIONS", "1500"))
_MEMORY = _os.getenv("ARC3_MEMORY", "1") == "1"
_CONTACT = _os.getenv("ARC3_CONTACT", "1") == "1"
_GAME_BUDGET_S = float(_os.getenv("ARC3_GAME_BUDGET_S", "30600"))
# Per-game CPU budget (seconds of *this thread's* CPU time). The rerun plays all 110 games as concurrent threads
# under one GIL, so wall time is shared and meaningless per game; thread CPU time is each game's fair-share meter.
_GAME_CPU_S = float(_os.getenv("ARC3_GAME_CPU_S", "700"))


class MyAgent(_FrameworkAgent):
    """ModelBasedAgent behind the framework's Agent interface. One instance per game."""

    MAX_ACTIONS = _MAX_ACTIONS

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.impl = ModelBasedAgent(game_id=self.game_id, seed=0, memory=_MEMORY, contact=_CONTACT)
        self._n = 0
        self._t0 = _time.time()
        self._c0 = None  # thread CPU clock, captured lazily inside the game's own thread (init runs in the main thread)

    @property
    def name(self) -> str:
        return f"{super().name}.modelbased.mem{int(_MEMORY)}.contact{int(_CONTACT)}"

    def is_done(self, frames, latest_frame) -> bool:
        if latest_frame.state is GameState.WIN:
            return True
        if _time.time() - _START > _BUDGET_S or _time.time() - self._t0 > _GAME_BUDGET_S:
            return True
        if self._c0 is None:
            self._c0 = _time.thread_time()
        if _time.thread_time() - self._c0 > _GAME_CPU_S:
            return True
        return self._n >= _MAX_ACTIONS

    def choose_action(self, frames, latest_frame) -> GameAction:
        self._n += 1
        if self._c0 is None:
            self._c0 = _time.thread_time()
        try:
            return self.impl.choose_action(frames, latest_frame)
        except Exception as e:  # never crash the swarm thread
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                return GameAction.RESET
            acts = [a for a in GameAction if a is not GameAction.RESET and a.value in (latest_frame.available_actions or [1, 2, 3, 4])]
            a = acts[self._n % len(acts)] if acts else GameAction.ACTION1
            if a.is_complex():
                a.set_data({"x": 32, "y": 32})
            return a


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Self-test in Kaggle's own environment: play two public games offline with the agent (no gateway needed)
    import sys, time, logging, types
    sys.path.insert(0, '/kaggle/working')
    fw = types.ModuleType('agents'); fa = types.ModuleType('agents.agent')
    class _A:
        MAX_ACTIONS = 80
        def __init__(self, card_id='', game_id='x', agent_name='', ROOT_URL='', record=False, arc_env=None, tags=None): self.game_id = game_id
        @property
        def name(self): return self.game_id
    fa.Agent = _A; fw.agent = fa; sys.modules['agents'] = fw; sys.modules['agents.agent'] = fa
    import my_agent
    import arc_agi
    from arc_agi import Arcade, OperationMode
    from arcengine import FrameData, GameState
    logging.getLogger().setLevel(logging.ERROR)
    arc = Arcade(environments_dir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files', operation_mode=OperationMode.OFFLINE)
    for gid in ['vc33', 'ls20']:
        env = arc.make(gid); ag = my_agent.MyAgent(game_id=gid); frames = [FrameData(levels_completed=0)]; t0 = time.time()
        for _ in range(150):
            a = ag.choose_action(frames, frames[-1]); raw = env.step(a, data=a.action_data.model_dump())
            if raw is None: break
            frames.append(FrameData(game_id=raw.game_id, frame=[x.tolist() for x in raw.frame], state=raw.state, levels_completed=raw.levels_completed, win_levels=raw.win_levels, available_actions=raw.available_actions))
            if frames[-1].state is GameState.WIN: break
        print(f'SELFTEST {gid}: actions {len(frames)-1} levels {frames[-1].levels_completed} secs {time.time()-t0:.1f} stats {ag.impl.stats().get("pred_acc")}')
        arc.close_scorecard()
    # produce a dummy submission file so "Save & Run All" succeeds
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
